# Statistical Analysis

## Introduction

Exploratory Data Analysis (EDA) identified several variables that appeared
to be associated with customer churn. However, observed differences and
patterns in a sample do not necessarily indicate statistically meaningful
relationships.

The objective of this analysis is to statistically evaluate the key
relationships identified during the exploratory and SQL analysis.

Different statistical methods will be selected based on the types of
variables being analyzed. Statistical significance will be evaluated using
a significance level of α = 0.05, while effect sizes will be used to assess
the practical strength of observed relationships.

The analysis focuses on determining:

1. Which categorical variables are statistically associated with churn?
2. How strong are those associations?
3. Do churned and retained customers differ significantly in numerical
   characteristics such as tenure and monthly charges?
4. Which findings provide the strongest statistical evidence for further
   business investigation?

Statistical significance will be interpreted separately from practical
significance, and the results will be treated as evidence of association
rather than proof of causation.

# Statistical Foundations

## 2.1 Hypothesis Testing

Statistical hypothesis testing provides a framework for evaluating whether
an observed pattern in a sample provides sufficient evidence of a
relationship in the population.

Each statistical test begins with two competing hypotheses:

- **Null hypothesis (H₀):** There is no statistically significant
  relationship or difference.
- **Alternative hypothesis (H₁):** There is a statistically significant
  relationship or difference.

The significance level for this analysis is:

**α = 0.05**

If the p-value is less than α, the null hypothesis will be rejected.

If the p-value is greater than or equal to α, there is insufficient
evidence to reject the null hypothesis.

A failure to reject H₀ does not prove that H₀ is true.

---

## 2.2 P-value

The p-value represents the probability of observing a result at least as
extreme as the observed result, assuming that the null hypothesis is true.

A small p-value provides evidence against the null hypothesis.

However, the p-value does not measure the size or practical importance
of an effect.

---

## 2.3 Effect Size

Effect size measures the magnitude or strength of an observed relationship
or difference.

This analysis uses:

- **Cramér's V** for categorical associations
- **Cohen's d** for differences between two numerical groups

Effect size is considered alongside statistical significance to avoid
treating statistically significant but practically negligible effects as
important business findings.

---

## 2.4 Association vs Causation

Statistical tests can identify evidence of association or differences
between groups. They do not, by themselves, establish that one variable
causes customer churn.

Therefore, findings in this analysis will be interpreted as associations
unless a causal research design provides stronger evidence.

In [4]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [23]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from src.statistics import (
    create_contingency_table,
    chi_square_test,
    cramers_v,
    welch_t_test,
    cohens_d,
    one_way_anova,
    eta_squared,
    pearson_correlation,
    spearman_correlation
)

from  statsmodels.stats.multicomp import pairwise_tukeyhsd

In [6]:
df = pd.read_excel(
    "/Users/ashwathimd/DataLab/Projects/customer-churn-analysis/data/processed/telco_customer_churn_clean.xlsx"
)

# Categorical Variable Analysis

## Business Question 30

### Is Contract Type statistically associated with Customer Churn?

#### Why are we testing this?

EDA and SQL analysis showed substantial differences in churn rates across
contract types. We now want to determine whether these differences provide
statistical evidence of an association between contract type and churn.

### Statistical Method

**Chi-Square Test of Independence**

- H₀: Contract Type and Churn are independent.
- H₁: Contract Type and Churn are associated.

If the association is statistically significant, **Cramér's V** will be
used to measure its strength.

In [7]:
contract_churn_table = create_contingency_table(
    df,
    "Contract",
    "Churn Value"
)
contract_churn_table

Churn Value,0,1
Contract,,
Month-to-month,2220,1655
One year,1307,166
Two year,1647,48


In [8]:
contract_chi_square = chi_square_test(
    contract_churn_table
)
contract_chi_square

{'chi2': np.float64(1184.5966),
 'p_value': np.float64(5.863038300673393e-258),
 'degrees_of_freedom': 2,
 'expected_frequencies': Churn Value               0            1
 Contract                                
 Month-to-month  2846.691751  1028.308249
 One year        1082.110180   390.889820
 Two year        1245.198069   449.801931}

In [9]:
contract_effect = cramers_v(
    contract_churn_table
)
contract_effect

{'cramers_v': np.float64(0.4101), 'effect_size': 'Moderate'}

### Statistical Insight

The Chi-Square Test produced a **χ² statistic of 1184.60** with **2 degrees of freedom** and a **p-value of 5.86 × 10⁻²⁵⁸**.

Since the p-value is substantially lower than the significance level (α = 0.05), we **reject the null hypothesis**. There is strong statistical evidence of an association between **Contract Type** and **Customer Churn**.

The **Cramér's V of 0.4101** indicates a **moderate association** between the two variables.

Combined with the observed churn rates from EDA — **42.71% for month-to-month, 11.27% for one-year, and 2.83% for two-year contracts** — the results indicate that contract type is an important factor associated with customer churn.

> **Important:** Statistical significance establishes an association, not causation. Therefore, the results do not imply that contract type directly causes customers to churn.


## Business Question 31

### Is Internet Service statistically associated with Customer Churn?

#### Objective

The Exploratory Data Analysis (EDA) and SQL analysis indicated noticeable differences in churn rates across different internet service types.

The objective of this analysis is to determine whether the observed differences provide statistically significant evidence of an association between **Internet Service** and **Customer Churn**.

The **Chi-Square Test of Independence** will be used to test whether the two categorical variables are statistically associated. If a significant association is identified, **Cramér's V** will be used to measure the strength of the relationship.

#### Hypotheses

- **H₀:** Internet Service and Customer Churn are independent.
- **H₁:** Internet Service and Customer Churn are associated.

The significance level for the test is:

**α = 0.05**

In [10]:
internet_churn_table = create_contingency_table(
    df,
    "Internet Service",
    "Churn Value"
)
internet_churn_table

Churn Value,0,1
Internet Service,,
DSL,1962,459
Fiber optic,1799,1297
No,1413,113


In [11]:
internet_chi_square = chi_square_test(
    internet_churn_table
)
internet_chi_square

{'chi2': np.float64(732.3096),
 'p_value': np.float64(9.571788222840544e-160),
 'degrees_of_freedom': 2,
 'expected_frequencies': Churn Value                 0           1
 Internet Service                         
 DSL               1778.539543  642.460457
 Fiber optic       2274.414880  821.585120
 No                1121.045577  404.954423}

In [12]:
internet_effect = cramers_v(
    internet_churn_table
)
internet_effect

{'cramers_v': np.float64(0.3225), 'effect_size': 'Moderate'}

### Statistical Insight

The Chi-Square Test produced a **χ² statistic of 732.31** with **2 degrees of freedom** and a **p-value of 9.57 × 10⁻¹⁶⁰**.

Since the p-value is substantially lower than the significance level (α = 0.05), we **reject the null hypothesis**. There is strong statistical evidence of an association between **Internet Service** and **Customer Churn**.

The **Cramér's V of 0.3225** indicates a **moderate association** between the two variables.

The observed churn rates show that **fiber-optic customers have the highest churn rate (41.9%)**, compared with **19.5% for DSL customers** and **7.4% for customers without internet service**.

These results suggest that Internet Service is an important factor associated with customer churn.

> **Important:** Statistical significance establishes an association, not causation. Therefore, these results do not imply that having fiber-optic service directly causes customers to churn.

## Business Question 32

### Is Payment Method statistically associated with Customer Churn?

#### Objective

The Exploratory Data Analysis (EDA) and SQL analysis indicated noticeable differences in churn rates across different payment methods, with **Electronic check** customers showing the highest churn rate.

The objective of this analysis is to determine whether the observed differences provide statistically significant evidence of an association between **Payment Method** and **Customer Churn**.

The **Chi-Square Test of Independence** will be used to determine whether Payment Method and Customer Churn are statistically associated. If a significant association is identified, **Cramér's V** will be used to measure the strength of the relationship.

#### Hypotheses

- **H₀:** Payment Method and Customer Churn are independent.
- **H₁:** Payment Method and Customer Churn are associated.

The significance level for the test is:

**α = 0.05**

In [13]:
payment_churn_table = create_contingency_table(
    df,
    "Payment Method",
    "Churn Value"
)

payment_churn_table

Churn Value,0,1
Payment Method,,
Bank transfer (automatic),1286,258
Credit card (automatic),1290,232
Electronic check,1294,1071
Mailed check,1304,308


In [14]:
payment_chi_square = chi_square_test(
    payment_churn_table
)

payment_chi_square

{'chi2': np.float64(648.1423),
 'p_value': np.float64(3.6823546520098007e-140),
 'degrees_of_freedom': 3,
 'expected_frequencies': Churn Value                          0           1
 Payment Method                                    
 Bank transfer (automatic)  1134.268919  409.731081
 Credit card (automatic)    1118.107057  403.892943
 Electronic check           1737.400256  627.599744
 Mailed check               1184.223768  427.776232}

In [15]:
payment_effect = cramers_v(
    payment_churn_table
)

payment_effect

{'cramers_v': np.float64(0.3034), 'effect_size': 'Moderate'}

### Statistical Insight

The Chi-Square Test produced a **χ² statistic of 648.14** with **3 degrees of freedom** and a **p-value of 3.68 × 10⁻¹⁴⁰**.

Since the p-value is substantially lower than the significance level (α = 0.05), we **reject the null hypothesis**. There is strong statistical evidence of an association between **Payment Method** and **Customer Churn**.

The **Cramér's V of 0.3034** indicates a **moderate association** between the two variables.

The observed churn rates show that **Electronic check customers have the highest churn rate (45.29%)**, substantially higher than customers using mailed check (19.11%), bank transfer (16.71%), or credit card (15.24%).

These results suggest that Payment Method is an important factor associated with customer churn, with **Electronic check customers representing a particularly high-risk segment**.

> **Important:** Statistical significance establishes an association, not causation. Therefore, these results do not imply that using electronic check directly causes customers to churn.

# Numerical Variable Analysis

Categorical analysis established that several categorical variables are statistically associated with customer churn.

We now examine whether **numerical variables differ significantly between customers who churned and those who remained with the company**.

Because Customer Churn divides customers into two independent groups, we can use an **Independent Samples t-test** to compare the mean of a numerical variable between the two groups.

However, statistical significance alone does not tell us how large the difference is. Therefore, **Cohen's d** will also be used to measure the magnitude of the difference.

## Business Question 33

### Do churned and non-churned customers have significantly different Monthly Charges?

#### Objective

The Exploratory Data Analysis (EDA) indicated that customers who churned tend to have different monthly charges compared with customers who remained with the company.

The objective of this analysis is to determine whether the difference in **mean Monthly Charges** between churned and non-churned customers is statistically significant.

An **Independent Samples t-test** will be used to test whether the two groups have significantly different mean Monthly Charges. **Cohen's d** will then be used to measure the practical magnitude of the difference.

#### Hypotheses

- **H₀:** The mean Monthly Charges are equal for churned and non-churned customers.
- **H₁:** The mean Monthly Charges are different for churned and non-churned customers.

The significance level for the test is:

**α = 0.05**

### Statistical Method

The two groups being compared are:

- **Non-churned customers:** Churn Value = 0
- **Churned customers:** Churn Value = 1

Welch's t-test evaluates whether the difference between their mean Monthly Charges is statistically significant.

Cohen's d measures the standardized magnitude of the difference between the two group means.

In [16]:
non_churned_charges = df.loc[
    df["Churn Value"] == 0,
    "Monthly Charges"
]

churned_charges = df.loc[
    df["Churn Value"] == 1,
    "Monthly Charges"
]

In [17]:
print(
    f"Non-churned customers: {len(non_churned_charges):,}"
)

print(
    f"Churned customers: {len(churned_charges):,}"
)

print(
    f"Non-churned mean: {non_churned_charges.mean():.2f}"
)

print(
    f"Churned mean: {churned_charges.mean():.2f}"
)

Non-churned customers: 5,174
Churned customers: 1,869
Non-churned mean: 61.27
Churned mean: 74.44


In [18]:
monthly_charges_test = welch_t_test(
    df,
    "Monthly Charges",
    "Churn Value"
)
monthly_charges_test

{'group_1': np.int64(1),
 'group_2': np.int64(0),
 'group_1_count': 1869,
 'group_2_count': 5174,
 'group_1_mean': np.float64(74.4413),
 'group_2_mean': np.float64(61.2651),
 't_statistic': np.float64(18.4075),
 'p_value': np.float64(8.592449331547065e-73),
 'degrees_of_freedom': np.float64(4135.795)}

In [19]:
monthly_charges_effect = cohens_d(
    df,
    "Monthly Charges",
    "Churn Value",
)
monthly_charges_effect

{'group_1': np.int64(1),
 'group_2': np.int64(0),
 'cohens_d': np.float64(0.4463),
 'effect_size': 'Small'}

### Statistical Insight

Welch's t-test produced a **t-statistic of 18.41** with a **p-value of 5.82 × 10⁻⁷³**.

Since the p-value is substantially lower than the significance level (**α = 0.05**), the null hypothesis is rejected.

Therefore, there is **strong statistical evidence that the mean Monthly Charges differ between churned and non-churned customers**.

Churned customers had a higher average Monthly Charge (**$74.44**) compared with non-churned customers (**$61.27**), representing a mean difference of approximately **$13.18**.

However, statistical significance does not necessarily imply a large practical effect. Cohen's d was **0.4463**, classified as a **small effect**.

This indicates that although Monthly Charges are statistically associated with churn, the magnitude of the difference between the two groups is relatively modest. Monthly Charges should therefore be considered alongside other customer characteristics when assessing churn risk.

## Business Question 34

### Do Monthly Charges differ significantly across Contract Types?

#### Objective

The EDA and SQL analysis indicated that customers on different contract types have different spending patterns.

The dataset contains three contract groups:

- Month-to-month
- One year
- Two year

The objective of this analysis is to determine whether the differences in **mean Monthly Charges** across these three contract groups are statistically significant.

Because we are comparing the mean of a numerical variable across **more than two independent groups**, a **One-Way ANOVA** will be used.

If the ANOVA indicates a statistically significant difference, **Eta Squared (η²)** will be used to measure the magnitude of the effect.

#### Hypotheses

- **H₀:** The mean Monthly Charges are equal across all Contract Types.
- **H₁:** At least one Contract Type has a different mean Monthly Charge.

The significance level for the test is:

**α = 0.05**

### Statistical Method

One-Way ANOVA compares the variability **between Contract Type group means** with the variability **within the groups**.

A statistically significant ANOVA result indicates that at least one group mean differs from the others.

However, ANOVA does not tell us which specific groups differ. If the ANOVA is significant, a post-hoc test would be required to identify the specific group differences.

In [20]:
contract_charges_summary = (
    df.groupby("Contract")["Monthly Charges"]
    .agg(
        customer_count = "count",
        mean_monthly_charge = "mean",
        std_monthly_charges = "std"
    )
    .round(2)
)
contract_charges_summary

,customer_count,mean_monthly_charge,std_monthly_charges
Contract,,,
Month-to-month,3875,66.40,26.93
One year,1473,65.05,31.84
Two year,1695,60.77,34.68


In [21]:
contract_anova = one_way_anova(
    df,
    "Monthly Charges",
    "Contract"
)
contract_anova

{'groups': ['Month-to-month', 'Two year', 'One year'],
 'group_means': {'Month-to-month': np.float64(66.3985),
  'Two year': np.float64(60.7704),
  'One year': np.float64(65.0486)},
 'f_statistic': np.float64(20.828),
 'p_value': np.float64(9.575270975922278e-10)}

In [22]:
contract_effect = eta_squared(
    df,
    "Monthly Charges",
    "Contract"
)
contract_effect

{'eta_squared': np.float64(0.0059), 'effect_size': 'Negligible'}

### Statistical Insight

The One-Way ANOVA produced an **F-statistic of 20.83** with a **p-value of 9.58 × 10⁻¹⁰**.

Since the p-value is substantially lower than the significance level (**α = 0.05**), the null hypothesis is rejected.

Therefore, there is **statistically significant evidence that mean Monthly Charges differ across Contract Types**.

The observed mean Monthly Charges were:

- **Month-to-month:** $66.40
- **One year:** $65.05
- **Two year:** $60.77

However, statistical significance does not necessarily indicate a meaningful practical difference. The calculated **η² = 0.0059**, indicating a **negligible effect**.

This means Contract Type explains approximately **0.59% of the variation in Monthly Charges**. Therefore, although the differences between contract groups are statistically detectable, Contract Type has very limited explanatory power for Monthly Charges.

>  ANOVA establishes that at least one group mean differs from the others, but it does not identify which specific groups are significantly different. A post-hoc analysis would be required to determine the individual group differences.

## Business Question 35

### Which Contract Types have significantly different Monthly Charges?

#### Objective

The One-Way ANOVA established that mean Monthly Charges differ significantly across Contract Types. However, ANOVA does not identify which specific contract groups are responsible for the observed difference.

The objective of this analysis is to determine which pairs of Contract Types have statistically significant differences in their mean Monthly Charges.

Because the ANOVA contains three independent groups, **Tukey's Honestly Significant Difference (HSD) test** will be used for pairwise comparisons while controlling the overall Type I error rate.

#### Hypotheses

For each pair of Contract Types:

- **H₀:** The two groups have equal mean Monthly Charges.
- **H₁:** The two groups have different mean Monthly Charges.

The significance level is:

**α = 0.05**

### Statistical Method

Tukey's HSD performs pairwise comparisons between all Contract Types following a significant One-Way ANOVA.

For each pair, the test provides:

- The mean difference between the groups
- The adjusted p-value
- A confidence interval for the mean difference
- Whether the difference is statistically significant

In [24]:
tukey_result = pairwise_tukeyhsd(
    endog=df["Monthly Charges"],
    groups=df["Contract"],
    alpha=0.05
)

print(tukey_result)

     Multiple Comparison of Means - Tukey HSD, FWER=0.05      
    group1      group2  meandiff p-adj   lower   upper  reject
--------------------------------------------------------------
Month-to-month One year  -1.3499 0.3056 -3.5029  0.8032  False
Month-to-month Two year  -5.6281    0.0 -7.6764 -3.5797   True
      One year Two year  -4.2782 0.0002 -6.7838 -1.7726   True
--------------------------------------------------------------


### Statistical Insight

Tukey's HSD test identified which Contract Type pairs have statistically significant differences in mean Monthly Charges.

The comparison between **Month-to-month and One year contracts** was not statistically significant (adjusted p-value = 0.3056). Therefore, we fail to reject the null hypothesis for this pair.

However, both comparisons involving **Two year contracts** were statistically significant:

- **Month-to-month vs Two year:** adjusted p-value < 0.001
- **One year vs Two year:** adjusted p-value = 0.0002

The mean Monthly Charges were:

- Month-to-month: **66.40**
- One year: **65.05**
- Two year: **60.77**

Therefore, Two year customers have significantly different average Monthly Charges from both Month-to-month and One year customers, while the difference between Month-to-month and One year customers is not statistically significant.

## Business Question 36

### Do customers who churn have significantly different Total Charges than customers who remain?

#### Objective

During the Exploratory Data Analysis (EDA), differences in Total Charges were observed between customers who churned and those who remained.

This analysis uses an **independent two-sample test** to determine whether the difference in mean Total Charges between the two groups is statistically significant.

Because customer groups may have different variances, the **Welch's t-test** will be used.

If a statistically significant difference is identified, **Cohen's d** will be used to measure the magnitude of the difference.

### Hypotheses

**Null Hypothesis (H₀):**

There is no difference in mean Total Charges between churned and retained customers.

**Alternative Hypothesis (H₁):**

There is a difference in mean Total Charges between churned and retained customers.

We use a significance level of:

**α = 0.05**

In [25]:
total_charges_test = welch_t_test(
    df,
    "Total Charges",
    "Churn Value"
)

total_charges_test

{'group_1': np.int64(1),
 'group_2': np.int64(0),
 'group_1_count': 1869,
 'group_2_count': 5163,
 'group_1_mean': np.float64(1531.7961),
 'group_2_mean': np.float64(2555.3441),
 't_statistic': np.float64(-18.8008),
 'p_value': np.float64(1.1524944112838777e-75),
 'degrees_of_freedom': np.float64(4042.9311)}

In [26]:
total_charges_effect = cohens_d(
    df,
    "Total Charges",
    "Churn Value"
)

total_charges_effect

{'group_1': np.int64(1),
 'group_2': np.int64(0),
 'cohens_d': np.float64(-0.4608),
 'effect_size': 'Small'}

### Statistical Insight

The Welch's t-test produced a **t-statistic of -18.80** with a **p-value of 1.15 × 10⁻⁷⁵**.

Since the p-value is substantially lower than the significance level (**α = 0.05**), we reject the null hypothesis.

This provides strong statistical evidence that **mean Total Charges differ between churned and retained customers**.

The mean Total Charges for churned customers (**1531.80**) were substantially lower than for retained customers (**2555.34**).

### Effect Size

Cohen's d was **-0.461**, indicating a **small effect** according to the interpretation used in this analysis.

The negative value indicates that the churned group had a lower mean Total Charges than the retained group.

Therefore, while the difference is statistically significant, the effect size indicates that the magnitude of the difference is relatively modest.

> **Important:** Statistical significance does not imply causation. Total Charges are also influenced by factors such as customer tenure, so this relationship should not be interpreted as evidence that lower Total Charges cause customer churn.

## Business Question 37

### Is Customer Tenure significantly different between churned and retained customers?

#### Objective

During the Exploratory Data Analysis (EDA), churn appeared to be more common among customers with shorter tenure.

This analysis will test whether the observed difference in customer tenure between churned and retained customers is statistically significant.

A **Welch's independent two-sample t-test** will be used because the two customer groups may have unequal variances.

If a statistically significant difference is found, **Cohen's d** will be used to measure the magnitude of the difference.

### Hypotheses

**Null Hypothesis (H₀):**

There is no difference in mean customer tenure between churned and retained customers.

**Alternative Hypothesis (H₁):**

There is a difference in mean customer tenure between churned and retained customers.

**Significance level: α = 0.05**

In [27]:
tenure_test = welch_t_test(
    df,
    "Tenure Months",
    "Churn Value"
)

tenure_test

{'group_1': np.int64(1),
 'group_2': np.int64(0),
 'group_1_count': 1869,
 'group_2_count': 5174,
 'group_1_mean': np.float64(17.9791),
 'group_2_mean': np.float64(37.57),
 't_statistic': np.float64(-34.8238),
 'p_value': np.float64(1.1954945472605766e-232),
 'degrees_of_freedom': np.float64(4048.2876)}

In [28]:
tenure_effect = cohens_d(
    df,
    "Tenure Months",
    "Churn Value"
)

tenure_effect

{'group_1': np.int64(1),
 'group_2': np.int64(0),
 'cohens_d': np.float64(-0.8522),
 'effect_size': 'Large'}

### Statistical Insight

The Welch's t-test produced a **t-statistic of -34.82** with a **p-value of 1.20 × 10⁻²³²**.

Since the p-value is substantially lower than the significance level (**α = 0.05**), we reject the null hypothesis.

This provides strong statistical evidence that **mean customer tenure differs between churned and retained customers**.

The mean tenure of churned customers was approximately **17.98 months**, compared with **37.57 months** for retained customers. Churned customers therefore had, on average, approximately **19.6 fewer months of tenure**.

### Effect Size

Cohen's d was **-0.8522**, indicating a **large effect**.

The negative value reflects the lower mean tenure of the churned group relative to the retained group.

Therefore, the difference in tenure is not only statistically significant but also **substantial in practical magnitude**.

> **Important:** This analysis establishes an association between customer tenure and churn, not causation. The shorter tenure observed among churned customers may also be related to other factors influencing customer churn.

## Business Question 38

### Is Customer Tenure significantly associated with Total Charges?

#### Objective

Previous analysis showed that churned customers have significantly lower Total Charges and substantially shorter tenure than retained customers.

Since Total Charges accumulate over the duration of a customer's relationship with the company, this analysis examines whether **Customer Tenure and Total Charges are statistically associated**.

A correlation analysis will be used to determine the strength and direction of this relationship.

Both **Pearson's correlation** and **Spearman's rank correlation** will be considered to assess the relationship.

### Hypotheses

**Null Hypothesis (H₀):**

There is no statistically significant correlation between Customer Tenure and Total Charges.

**Alternative Hypothesis (H₁):**

There is a statistically significant correlation between Customer Tenure and Total Charges.

**Significance level: α = 0.05**

In [32]:
tenure_total_pearson = pearson_correlation(
    df["Tenure Months"],
    df["Total Charges"]
)

tenure_total_pearson

{'correlation': np.float64(0.8259), 'p_value': np.float64(0.0)}

In [33]:
tenure_total_spearman = spearman_correlation(
    df["Tenure Months"],
    df["Total Charges"]
)

tenure_total_spearman

{'correlation': np.float64(0.8892), 'p_value': np.float64(0.0)}

### Statistical Insight

Both Pearson's and Spearman's correlation tests indicate a statistically significant positive association between **Customer Tenure** and **Total Charges**.

The Pearson correlation coefficient was **0.8259**, indicating a strong positive linear relationship, while the Spearman correlation coefficient was **0.8892**, indicating a very strong positive monotonic relationship.

Both p-values were below the significance level (**α = 0.05**), providing strong evidence against the null hypothesis of no correlation.

Therefore, customers with longer tenure tend to have substantially higher Total Charges.

### Relationship to Previous Findings

This finding provides important context for the result observed in **Business Question 36**, where churned customers were found to have significantly lower Total Charges.

Since churned customers also have substantially shorter tenure, the lower Total Charges observed among churned customers may be partly explained by their shorter time as customers. Total Charges accumulate over the duration of the customer relationship.

Therefore, **Total Charges should not be interpreted as an independent causal driver of churn based on this analysis alone**.

> **Important:** Correlation does not establish causation. The strong relationship between Tenure and Total Charges is expected to some extent because Total Charges accumulate over time.

## Business Question 39

### Is Monthly Charges significantly associated with Total Charges?

#### Objective

Previous analysis established that Customer Tenure has a strong positive association with Total Charges.

This analysis examines whether **Monthly Charges** are also associated with **Total Charges**.

Pearson's and Spearman's correlation tests will be used to evaluate the strength, direction, and statistical significance of the relationship.

### Hypotheses

**Null Hypothesis (H₀):**

There is no statistically significant correlation between Monthly Charges and Total Charges.

**Alternative Hypothesis (H₁):**

There is a statistically significant correlation between Monthly Charges and Total Charges.

**Significance level: α = 0.05**

In [36]:
monthly_total_pearson = pearson_correlation(
    df["Monthly Charges"],
    df["Total Charges"]
)

monthly_total_pearson

{'correlation': np.float64(0.6511), 'p_value': np.float64(0.0)}

In [37]:
monthly_total_spearman = spearman_correlation(
    df["Monthly Charges"],
    df["Total Charges"]
)

monthly_total_spearman

{'correlation': np.float64(0.638), 'p_value': np.float64(0.0)}

### Statistical Insight

Both Pearson's and Spearman's correlation tests indicate a strong positive association between **Monthly Charges** and **Total Charges**.

- **Pearson correlation:** r = 0.6511
- **Spearman correlation:** ρ = 0.6380
- **p-value:** < 0.001

Since the p-value is substantially below the significance level (α = 0.05), the null hypothesis is rejected.

Therefore, there is a **statistically significant positive correlation** between Monthly Charges and Total Charges.

However, this relationship should be interpreted cautiously. Total Charges represent the accumulated amount paid by a customer, meaning that customers with higher monthly charges will generally accumulate total charges at a faster rate. Therefore, the observed correlation is expected from the structure of these variables and should not be interpreted as evidence that Monthly Charges independently cause higher Total Charges.

In [38]:
tenure_monthly_pearson = pearson_correlation(
    df["Tenure Months"],
    df["Monthly Charges"]
)

tenure_monthly_pearson

{'correlation': np.float64(0.2479),
 'p_value': np.float64(4.094044991493961e-99)}

In [39]:
tenure_monthly_spearman = spearman_correlation(
    df["Tenure Months"],
    df["Monthly Charges"]
)

tenure_monthly_spearman

{'correlation': np.float64(0.2764),
 'p_value': np.float64(1.0271266876409566e-123)}

### Statistical Insight

Both Pearson's and Spearman's correlation tests indicate a statistically significant positive relationship between **Customer Tenure** and **Monthly Charges**.

- **Pearson correlation:** r = 0.2479
- **Spearman correlation:** ρ = 0.2764
- **p-value:** < 0.001 for both tests

Since both p-values are substantially below the significance level (α = 0.05), the null hypothesis is rejected.

However, the correlation coefficients indicate that the relationship is **weak**. Therefore, although customers with longer tenure tend to have somewhat higher Monthly Charges, tenure explains only a limited amount of variation in Monthly Charges.

This is an example of the distinction between **statistical significance and practical strength**: the relationship is statistically significant, but its magnitude is relatively small.

## Business Question 41

### Do Monthly Charges differ significantly across Internet Service types?

#### Objective

Internet Service was found to have a statistically significant association with Customer Churn.

This analysis examines whether customers using different Internet Service types also have significantly different Monthly Charges.

A **One-Way ANOVA** will be used to determine whether the mean Monthly Charges differ across Internet Service groups.

If a statistically significant difference is found, **Eta Squared (η²)** will be used to measure the effect size, followed by **Tukey's HSD** to identify which specific groups differ from one another.

The significance level is set at **α = 0.05**.

### Hypotheses

**Null Hypothesis (H₀):**

The mean Monthly Charges are equal across all Internet Service types.

**Alternative Hypothesis (H₁):**

At least one Internet Service type has a different mean Monthly Charge.

**Significance level:** α = 0.05

In [40]:
internet_monthly_anova = one_way_anova(
    df,
    "Monthly Charges",
    "Internet Service"
)

internet_monthly_anova

{'groups': ['DSL', 'Fiber optic', 'No'],
 'group_means': {'DSL': np.float64(58.1022),
  'Fiber optic': np.float64(91.5001),
  'No': np.float64(21.0792)},
 'f_statistic': np.float64(16111.6463),
 'p_value': np.float64(0.0)}

In [41]:
internet_monthly_effect = eta_squared(
    df,
    "Monthly Charges",
    "Internet Service"
)

internet_monthly_effect

{'eta_squared': np.float64(0.8207), 'effect_size': 'Large'}

### Statistical Insight

The One-Way ANOVA produced an **F-statistic of 16,111.65** with a p-value < 0.001.

Since the p-value is substantially below the significance level (α = 0.05), the null hypothesis is rejected.

Therefore, there is a **statistically significant difference in mean Monthly Charges across Internet Service types**.

The group means are:

- **DSL:** 58.10
- **Fiber optic:** 91.50
- **No internet service:** 21.08

The calculated **η² = 0.8207**, indicating a **large effect**. This suggests that Internet Service explains a substantial proportion of the variation in Monthly Charges.

However, ANOVA only establishes that at least one group differs from the others. A post-hoc **Tukey HSD test** will therefore be used to determine which specific Internet Service groups have statistically significant differences in Monthly Charges.

In [42]:
internet_monthly_tukey = pairwise_tukeyhsd(
    endog=df["Monthly Charges"],
    groups=df["Internet Service"],
    alpha=0.05
)

print(internet_monthly_tukey)

      Multiple Comparison of Means - Tukey HSD, FWER=0.05      
   group1      group2   meandiff p-adj  lower    upper   reject
---------------------------------------------------------------
        DSL Fiber optic   33.398   0.0  32.5875  34.2084   True
        DSL          No  -37.023   0.0 -37.9994 -36.0466   True
Fiber optic          No -70.4209   0.0 -71.3553 -69.4866   True
---------------------------------------------------------------


### Tukey HSD Results

The Tukey HSD post-hoc test was performed because the One-Way ANOVA identified a statistically significant difference across Internet Service groups.

All pairwise comparisons were statistically significant after adjustment for multiple comparisons:

- **DSL vs Fiber optic:** mean difference = 33.40, p < 0.001
- **DSL vs No internet service:** mean difference = 37.02, p < 0.001
- **Fiber optic vs No internet service:** mean difference = 70.42, p < 0.001

Therefore, **all three Internet Service groups have statistically different mean Monthly Charges**.

The mean Monthly Charges follow a clear pattern:

**Fiber optic ($91.50) > DSL ($58.10) > No internet service ($21.08)**.

Combined with the ANOVA result and the large η² effect size (0.8207), this indicates that Internet Service type is strongly associated with differences in Monthly Charges.

This finding provides useful context for the earlier analysis showing a significant association between Internet Service and Customer Churn. However, this analysis alone does not establish that pricing differences cause churn.

## Business Question 42

### Do Monthly Charges differ significantly across Payment Methods?

#### Objective

Payment Method was found to have a statistically significant association with Customer Churn.

This analysis examines whether customers using different Payment Methods also have significantly different Monthly Charges.

A **One-Way ANOVA** will be used to determine whether the mean Monthly Charges differ across Payment Method groups.

If a statistically significant difference is found, **Eta Squared (η²)** will be used to measure the effect size. A **Tukey HSD** post-hoc test will then be used to identify which specific Payment Method groups differ significantly.

The significance level is set at **α = 0.05**.

### Hypotheses

**Null Hypothesis (H₀):**

The mean Monthly Charges are equal across all Payment Method groups.

**Alternative Hypothesis (H₁):**

At least one Payment Method group has a different mean Monthly Charge.

**Significance level:** α = 0.05

In [43]:
payment_monthly_anova = one_way_anova(
    df,
    "Monthly Charges",
    "Payment Method"
)

payment_monthly_anova

{'groups': ['Mailed check',
  'Electronic check',
  'Bank transfer (automatic)',
  'Credit card (automatic)'],
 'group_means': {'Mailed check': np.float64(43.9171),
  'Electronic check': np.float64(76.2558),
  'Bank transfer (automatic)': np.float64(67.1926),
  'Credit card (automatic)': np.float64(66.5124)},
 'f_statistic': np.float64(450.319),
 'p_value': np.float64(1.1802197193655633e-267)}

In [44]:
payment_monthly_effect = eta_squared(
    df,
    "Monthly Charges",
    "Payment Method"
)

payment_monthly_effect

{'eta_squared': np.float64(0.161), 'effect_size': 'Large'}

### Statistical Insight

The One-Way ANOVA produced an **F-statistic of 450.32** with a p-value < 0.001.

Since the p-value is substantially below the significance level (α = 0.05), the null hypothesis is rejected.

Therefore, there is a **statistically significant difference in mean Monthly Charges across Payment Method groups**.

The group means are:

- **Mailed check:** 43.92
- **Bank transfer (automatic):** 67.19
- **Credit card (automatic):** 66.51
- **Electronic check:** 76.26

The calculated **η² = 0.161**, indicating a **large effect**. This suggests that Payment Method is associated with a meaningful proportion of the variation in Monthly Charges.

However, ANOVA only establishes that at least one group differs from another. A post-hoc **Tukey HSD test** will therefore be used to identify which specific Payment Method groups have statistically significant differences in Monthly Charges.

In [45]:
payment_monthly_tukey = pairwise_tukeyhsd(
    endog=df["Monthly Charges"],
    groups=df["Payment Method"],
    alpha=0.05
)

print(payment_monthly_tukey)

                   Multiple Comparison of Means - Tukey HSD, FWER=0.05                    
          group1                   group2         meandiff p-adj   lower    upper   reject
------------------------------------------------------------------------------------------
Bank transfer (automatic) Credit card (automatic)  -0.6803 0.9035   -3.239   1.8784  False
Bank transfer (automatic)        Electronic check   9.0632    0.0   6.7455  11.3809   True
Bank transfer (automatic)            Mailed check -23.2756    0.0 -25.7981 -20.7531   True
  Credit card (automatic)        Electronic check   9.7434    0.0   7.4156  12.0712   True
  Credit card (automatic)            Mailed check -22.5953    0.0 -25.1271 -20.0636   True
         Electronic check            Mailed check -32.3388    0.0 -34.6267 -30.0508   True
------------------------------------------------------------------------------------------


### Tukey HSD Results

The Tukey HSD post-hoc test was used to identify which Payment Method groups differ significantly in mean Monthly Charges.

The results show that:

- **Bank transfer vs. Credit card:** No statistically significant difference.
- **Bank transfer vs. Electronic check:** Significant difference.
- **Bank transfer vs. Mailed check:** Significant difference.
- **Credit card vs. Electronic check:** Significant difference.
- **Credit card vs. Mailed check:** Significant difference.
- **Electronic check vs. Mailed check:** Significant difference.

Therefore, **Bank transfer and Credit card customers have statistically similar mean Monthly Charges**, while Electronic Check and Mailed Check customers differ significantly from the other payment groups.

Combined with the group means, Electronic Check customers have the highest average Monthly Charges (76.26), while Mailed Check customers have the lowest (43.92).

# Q43 — Statistical Evidence Summary

## Objective

The statistical tests performed above evaluated whether the patterns identified during Exploratory Data Analysis are statistically significant and, where appropriate, measured the strength of the observed relationships using effect sizes.

This section consolidates the findings across all analyses to identify the variables with the strongest statistical evidence of association with customer churn.

Rather than ranking variables solely by p-value, the interpretation will prioritize **effect size and practical significance**, since large datasets can produce statistically significant results even when the underlying effect is small.

## Summary of Statistical Evidence

The following table summarizes the statistical evidence obtained throughout the analysis.

| Variable / Relationship | Statistical Test | p-value | Effect Size | Interpretation |
|---|---|---:|---:|---|
| Contract → Churn | Chi-Square | < 0.001 | Cramér's V = 0.4101 | Moderate association |
| Internet Service → Churn | Chi-Square | < 0.001 | Cramér's V = 0.3225 | Moderate association |
| Payment Method → Churn | Chi-Square | < 0.001 | Cramér's V = 0.3034 | Moderate association |
| Monthly Charges → Churn | Welch's t-test | < 0.001 | Cohen's d = 0.4463 | Small effect |
| Total Charges → Churn | Welch's t-test | < 0.001 | Cohen's d = -0.4608 | Small effect |
| Tenure → Churn | Welch's t-test | < 0.001 | Cohen's d = -0.8522 | Large effect |
| Contract → Monthly Charges | One-Way ANOVA | < 0.001 | η² = 0.0059 | Negligible effect |
| Internet Service → Monthly Charges | One-Way ANOVA | < 0.001 | η² = 0.8207 | Large effect |
| Payment Method → Monthly Charges | One-Way ANOVA | < 0.001 | η² = 0.1610 | Large effect |
| Tenure ↔ Total Charges | Pearson / Spearman | < 0.001 | r = 0.8259 / ρ = 0.8892 | Strong positive relationship |
| Monthly Charges ↔ Total Charges | Pearson / Spearman | < 0.001 | r = 0.6511 / ρ = 0.6380 | Moderate–strong positive relationship |
| Tenure ↔ Monthly Charges | Pearson / Spearman | < 0.001 | r = 0.2479 / ρ = 0.2764 | Weak positive relationship |

## Key Findings

### 1. Strongest churn-related evidence

Among the numerical variables examined, **Tenure Months has the largest effect size**, with Cohen's d = -0.8522. Churned customers have substantially lower average tenure than non-churned customers.

This indicates that customer tenure has a strong association with churn in this dataset.

### 2. Moderate categorical associations

Contract Type, Internet Service, and Payment Method all show statistically significant associations with Customer Churn, with Cramér's V values of 0.4101, 0.3225, and 0.3034 respectively.

These represent moderate associations, making these variables important candidates for further business investigation.

### 3. Pricing variables show smaller churn effects

Monthly Charges and Total Charges are both statistically significantly different between churned and non-churned customers. However, their Cohen's d values indicate relatively small effects.

Therefore, statistical significance alone should not be interpreted as evidence that pricing is one of the strongest determinants of churn.

### 4. Pricing differences are strongly associated with service and payment structure

Internet Service and Payment Method show large effects on Monthly Charges, with η² values of 0.8207 and 0.1610 respectively.

This indicates that differences in Monthly Charges are strongly related to the service and payment structures customers use.

Contract Type, in contrast, has a statistically significant but negligible effect on Monthly Charges (η² = 0.0059).

### 5. Correlations involving Total Charges require careful interpretation

Tenure and Total Charges show a very strong positive correlation, while Monthly Charges and Total Charges also show a substantial positive correlation.

These relationships are partly expected from the construction of Total Charges: customers who remain subscribed for longer have more time to accumulate charges, while customers with higher monthly charges accumulate Total Charges more quickly.

Therefore, these correlations should not be interpreted as evidence of an independent causal relationship.

## Overall Statistical Conclusion

The statistical analysis provides evidence that several customer characteristics are associated with churn.

The strongest evidence among the variables directly evaluated against churn comes from **Tenure**, followed by the categorical variables **Contract Type, Internet Service, and Payment Method**.

Although Monthly Charges and Total Charges are statistically significant, their effect sizes are considerably smaller. This demonstrates why statistical significance and practical significance must be evaluated together.

Overall, the results suggest that customer retention is more strongly associated with **customer tenure and service/contract characteristics** than with the absolute amount charged to customers.

These findings will be used in the final section to develop business-oriented conclusions and recommendations.

# Q44 — Business Conclusions & Recommendations

## Objective

The purpose of the statistical analysis was not only to determine whether relationships are statistically significant, but to identify which factors have meaningful associations with customer churn.

Based on the statistical evidence obtained throughout this analysis, this section translates the findings into actionable business conclusions and recommendations.

The conclusions below distinguish between **statistical significance**, **effect size**, and **business relevance**. Statistical association does not imply causation.

## Key Business Findings

### 1. Customer Tenure is the strongest churn-related factor

Tenure showed the largest effect size among the numerical variables analyzed, with a Cohen's d of **-0.8522**.

Churned customers had a substantially lower average tenure (**17.98 months**) compared with non-churned customers (**37.57 months**).

This indicates that customers who churn tend to leave relatively early in their relationship with the company.

**Business implication:** Customer retention efforts should place particular emphasis on the early stages of the customer lifecycle.

### 2. Contract Type is strongly associated with churn

Contract Type showed a statistically significant association with churn, with a **Cramér's V of 0.4101**, representing a moderate association.

The exploratory and statistical analysis indicates substantially higher churn among **Month-to-month customers** compared with customers on one-year and two-year contracts.

**Business implication:** The company should investigate strategies for encouraging customers to move from month-to-month contracts toward longer-term plans, provided that such offers remain attractive to customers.

### 3. Internet Service is associated with customer churn

Internet Service showed a statistically significant association with churn, with a **Cramér's V of 0.3225**, representing a moderate association.

Fiber optic customers showed substantially higher churn than DSL and customers without internet service.

The supporting pricing analysis also found a very large effect of Internet Service on Monthly Charges (**η² = 0.8207**).

**Business implication:** Fiber optic customers should be investigated further to determine whether pricing, service quality, customer expectations, technical issues, or other factors may contribute to their higher churn rate.

The statistical analysis identifies the association, but does not establish which underlying factor causes the higher churn.

### 4. Payment Method is associated with churn

Payment Method showed a statistically significant association with churn, with a **Cramér's V of 0.3034**, representing a moderate association.

Electronic check customers showed substantially higher churn than customers using the other payment methods.

The supporting pricing analysis also found statistically significant differences in Monthly Charges across payment methods.

**Business implication:** Electronic check customers represent a segment that may warrant targeted retention analysis. The company should investigate whether payment friction, customer characteristics, pricing, or other factors explain the observed difference in churn.

### 5. Monthly Charges are statistically significant but have a smaller churn effect

Monthly Charges were significantly different between churned and non-churned customers, but the effect size was relatively small (**Cohen's d = 0.4463**).

Churned customers had a higher average Monthly Charge (**74.44**) than non-churned customers (**61.27**).

Therefore, while pricing appears to be relevant, the effect size suggests that Monthly Charges alone do not explain customer churn.

**Business implication:** Pricing should be considered alongside contract type, service type, tenure, and other customer characteristics rather than treated as an independent explanation for churn.

### 6. Total Charges should not be treated as a primary churn driver

Total Charges were statistically different between churned and non-churned customers, but the effect size was small (**Cohen's d = -0.4608**).

Furthermore, Total Charges are accumulated over the customer's tenure. Therefore, the relationship between Total Charges and churn is partly influenced by how long a customer has remained with the company.

**Business implication:** Total Charges are useful for describing customer value and history, but should not be interpreted as evidence that accumulated spending itself causes churn.

## Recommended Business Actions

Based on the combined statistical evidence, the following areas should be prioritized:

1. **Focus on early-stage customer retention**
   - Identify customers at risk during the early months of their relationship.
   - Develop onboarding and early engagement strategies.
   - Monitor early-tenure churn closely.

2. **Investigate month-to-month customers**
   - Identify why month-to-month customers churn at higher rates.
   - Evaluate incentives for migration to longer-term contracts.
   - Ensure that retention offers are financially sustainable.

3. **Investigate Fiber optic customer churn**
   - Examine service quality, pricing, technical issues, and customer expectations.
   - Segment Fiber optic customers further to identify high-risk subgroups.

4. **Investigate Electronic Check customers**
   - Determine whether payment friction or other customer characteristics are associated with their higher churn.
   - Evaluate whether alternative payment options or targeted interventions could improve retention.

5. **Avoid relying on a single churn indicator**
   - Customer tenure, contract type, internet service, payment method, and pricing should be evaluated together.
   - Statistical significance alone should not determine business priority; effect size and business context should also be considered.

## Final Conclusion

The statistical analysis confirms that customer churn is significantly associated with several customer characteristics.

Among the variables directly evaluated against churn, **Tenure shows the strongest numerical effect**, while **Contract Type, Internet Service, and Payment Method show moderate categorical associations**.

Monthly Charges and Total Charges are also statistically significant, but their smaller effect sizes indicate that they are weaker churn-related factors when considered individually.

The analysis therefore suggests that customer retention efforts should prioritize **early-tenure customers and customer segments defined by contract, internet service, and payment method**, while pricing should be considered as part of a broader customer profile rather than as a standalone explanation for churn.

These findings provide a statistical foundation for the recommendations developed in the project. However, because the analysis is observational, the identified relationships should be interpreted as **associations rather than causal effects**. Further analysis using predictive modeling and controlled experiments would be required to establish causality and estimate the potential impact of specific retention interventions.